# Fraud Detection

## Overview

In this demo, we show how user profile attributes (address, phone number, and email) can be analyzed to find uncommon sharing patterns.

Most users typically have unique identifiers. Small amounts of sharing can be normal, but suspicious structures appear when users share different identifiers across different neighbors (for example, sharing an email with one account and a phone with another).

This notebook keeps the original fraud-detection logic and updates the implementation to the **v1 PyRel API**.

***

## Let's get started!

### Importing Packages

We start by installing and importing all the Python packages and modules that we will need, including the `relationalai` package. We then create a sample in-memory data set.

In [ ]:
%pip install -U "relationalai==1.0.0"

In [ ]:
import pandas as pd

from relationalai.semantics import Integer, Model, String
from relationalai.semantics.std import aggregates
from relationalai.semantics.reasoners import graph

In [ ]:
users_data = [
    {"id": 1, "fullname": "John Doe", "phone_number": "123-456-7890", "email": "john.doe@example.com", "address_id": 1, "credit_card_number": "4111111111111111"},
    {"id": 2, "fullname": "Jane Smith", "phone_number": "123-456-7891", "email": "weird.email@example.com", "address_id": 2, "credit_card_number": "5500000000000004"},
    {"id": 3, "fullname": "Bob Brown", "phone_number": "123-456-7893", "email": "bob.brown@example.com", "address_id": 3, "credit_card_number": "4111111111111112"},
    {"id": 4, "fullname": "David Evans", "phone_number": "123-456-7896", "email": "weird.email@example.com", "address_id": 1, "credit_card_number": "5500000000000005"},
    {"id": 5, "fullname": "Eva Green", "phone_number": "123-456-7896", "email": "eva.green@example.com", "address_id": 2, "credit_card_number": "340000000000010"},
    {"id": 6, "fullname": "Grace White", "phone_number": "123-456-7898", "email": "grace.white@example.com", "address_id": 3, "credit_card_number": "5500000000000006"},
    {"id": 7, "fullname": "Hannah Lee", "phone_number": "123-456-7899", "email": "hannah.lee@example.com", "address_id": 1, "credit_card_number": "340000000000011"},
    {"id": 8, "fullname": "Jack Wilson", "phone_number": "222-333-4444", "email": "jack.wilson@example.com", "address_id": 4, "credit_card_number": "4111111111111114"},
    {"id": 9, "fullname": "Kathy Brown", "phone_number": "333-444-5555", "email": "kathy.brown@example.com", "address_id": 5, "credit_card_number": "5500000000000007"},
]

addresses_data = [
    {"id": 1, "street_address": "123 Fake St", "city": "Springfield", "state": "IL"},
    {"id": 2, "street_address": "456 Elm St", "city": "Springfield", "state": "IL"},
    {"id": 3, "street_address": "123 Oak St", "city": "Springfield", "state": "IL"},
    {"id": 4, "street_address": "678 Pine St", "city": "Springfield", "state": "IL"},
    {"id": 5, "street_address": "890 Cedar St", "city": "Springfield", "state": "IL"},
]

### Defining the Model

We define concepts and relationships for `User`, `Address`, `CreditCard`, `Phone`, and `Email`, then derive suspicious activity signals using graph and rules-based reasoning.

In [ ]:
m = Model("FraudDetection")

Address = m.Concept("Address", identify_by={"id": Integer})
Address.street_address = m.Property(f"{Address} has street address {String:street_address}")
Address.city = m.Property(f"{Address} is in city {String:city}")
Address.state = m.Property(f"{Address} is in state {String:state}")

User = m.Concept("User", identify_by={"id": Integer})
User.fullname = m.Property(f"{User} is named {String:fullname}")
User.phone_number = m.Property(f"{User} has phone number {String:phone_number}")
User.email = m.Property(f"{User} has email {String:email}")
User.credit_card_number = m.Property(f"{User} has credit card number {String:credit_card_number}")
User.address = m.Relationship(f"{User} lives at {Address:address}")

`Address` and `User` are *entity types* that are identified by an `id` property.

## Load Data Into the Model

Now we load data into the model. In this example, we load the DataFrames we created early into the model using `m.data(...)` to create a table-like structure and then define how the data maps to the concepts and relationships in our model.

In [ ]:
addresses = m.data(addresses_data)
users = m.data(users_data)

m.define(Address.new(addresses.to_schema()))
m.define(
    User.new(
        users.to_schema(exclude=["address_id"]),
        address=Address.filter_by(id=users.address_id),
    )
)

> **Note**. If you want this notebook to run directly against Snowflake tables, you can replace `m.data(...)` with `m.Table("MY_DB.MY_SCHEMA.MY_TABLE")`. As long as the tables have the same schema as the DataFrames, the same model definitions and reasoning will work without modification.


## Getting to know the input data

Let's query users and their linked entities.

Each `User` now links to an `Address`.

In [ ]:
m.select(
    User.fullname,
    User.address.street_address,
    User.credit_card,
    User.email,
    User.phone,
).to_df()

## Applying Graph algorithm

To detect uncommon sharing patterns, we identify connected user communities in the identity graph.

We use Weakly Connected Components to assign each user to a community, then inspect group composition.

In [ ]:
g = graph.Graph(m, directed=False, weighted=False)

m.define(
    g.Edge.new(src=User, dst=User.has_address),
    g.Edge.new(src=User, dst=User.has_credit_card),
    g.Edge.new(src=User, dst=User.has_phone),
    g.Edge.new(src=User, dst=User.has_email),
)

User.community = g.weakly_connected_component(of=User)

### How many user groups were found?

Let's list users by connected community to inspect group size and composition.

In [ ]:
groups_df = m.select(User.fullname, User.community.alias("community")).to_df()
for i, (community, df) in enumerate(groups_df.groupby("community"), start=1):
    print(f"Group {i} with {len(df)} connected users: {sorted(df['fullname'].tolist())}")

> **Tip**. You can already see one community is uncommonly large, which is a useful prior before running explicit suspicious-user rules.

## Rule-based detection of uncommon patterns

Now that we've identified all the groups of users in our graph, let's add some rules to automatically detect groups and users in them that show unusual behavior.

First, we can identify groups that are uncommonly large, let's say, having 4 or more users. We create a new type called `LargeGroupUser` to mark users who belong to such groups.

In [ ]:
LARGE_GROUP_SIZE = 4

LargeGroupUser = m.Concept("LargeGroupUser", extends=[User])

users_per_community = aggregates.count(User).per(User.community)
m.define(LargeGroupUser(User)).where(users_per_community >= LARGE_GROUP_SIZE)

Next, let's take a closer look at the marked users.
If we see among them someone sharing email or phone number, but at the same time living in separate places, we can say it is an example of suspicious behavior.

We again create a new `SuspiciousUser` type and write a rule to detect users to set it for.

In [ ]:
SuspiciousUser = m.Concept("SuspiciousUser", extends=[User])

user1 = LargeGroupUser.ref()
user2 = LargeGroupUser.ref()

m.define(SuspiciousUser(user1)).where(
    LargeGroupUser(user1),
    LargeGroupUser(user2),
    user1 != user2,
    user1.community == user2.community,
    user1.has_address != user2.has_address,
    (user1.has_email == user2.has_email) | (user1.has_phone == user2.has_phone),
)

Lastly, we want to mark as suspicious users, who share physical address with another suspicious user.

In [ ]:
m.define(SuspiciousUser(User)).where(
    User.address == SuspiciousUser.address,
)

### Query for the results

Below, we list identified suspicious users and the key linked attributes used for investigation.

In [ ]:
suspicious_df = (
    m.select(
        SuspiciousUser.id,
        SuspiciousUser.fullname,
        SuspiciousUser.has_credit_card.number.alias("credit_card_number"),
        SuspiciousUser.has_address.street_address.alias("street_address"),
    )
    .to_df()
    .sort_values("id")
    .reset_index(drop=True)
)
suspicious_df

## Writing results back to Snowflake

As a final step, we export identified suspicious users (with selected details) into a Snowflake table using the `into(...).exec()` pattern.

In [ ]:
destination = m.Table("RAI_DEMO.FRAUD_DETECTION.SUSPICIOUS_USERS_V1")

(
    m.select(
        SuspiciousUser.id,
        SuspiciousUser.fullname,
        SuspiciousUser.has_credit_card.number.alias("credit_card_number"),
        SuspiciousUser.has_address.street_address.alias("street_address"),
    )
    .into(destination)
    .exec()
)

Let's inspect the destination table.

In [ ]:
# Get a Snowpark session and query the results back from Snowflake
session = m.config.get_session()

session.sql(
    "SELECT * FROM RAI_DEMO.FRAUD_DETECTION.SUSPICIOUS_USERS_V1"
).to_pandas()